# CS55 Production Evidence Evaluation

Human-reviewed panel extraction and global Final-led matching.
The notebook calls the same modules as `run_demo.py`. Automatic matches remain provisional.

## 1. Project and configuration

Set `PROJECT_ROOT` only if this notebook is opened outside the packaged project.
Copy `config.example.json` to `config.local.json`, then enter your file paths.
All relative material paths are resolved relative to the configuration file.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display, Image as DisplayImage

# If this notebook is outside the project, enter the CS55_system directory here.
PROJECT_ROOT = None
if PROJECT_ROOT is None:
    candidates = [Path.cwd(), *Path.cwd().parents]
    PROJECT_ROOT = next((p for p in candidates if (p / "src/cs55_demo/system.py").is_file()), None)
if PROJECT_ROOT is None:
    raise ValueError("Set PROJECT_ROOT to the CS55_system project directory")
PROJECT_ROOT = Path(PROJECT_ROOT).expanduser().resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from cs55_demo.system import (load_config, prepare, retrieve, inspect_provenance,
                              apply_panel_review, apply_reviews, save_json)

CONFIG_PATH = PROJECT_ROOT / "config.local.json"
if not CONFIG_PATH.exists():
    raise FileNotFoundError("Copy config.example.json to config.local.json and enter your material paths first")
config = load_config(CONFIG_PATH)
display(pd.DataFrame([config.get("sampling", {})]))

## 2. Review panel extraction and video samples

Automatic detection proposes panel boundaries but does not verify them. Review every page,
edit bounding boxes or their order, and mark each candidate `accept` or `reject`.
You may append a manual candidate using rendered-page pixel coordinates.
Video start/end times and sample intervals are configured independently for Final and Animatic.

In [ ]:
prepared = prepare(config)
RUN_DIR = Path(prepared["run_dir"])
manifest = json.loads((RUN_DIR / "panel_manifest.json").read_text()) if (RUN_DIR / "panel_manifest.json").exists() else None
if manifest:
    for page in manifest["pages"]:
        print(f"Page {page['page_index']}: {page['preview_path']}")
        display(DisplayImage(filename=page["preview_path"], width=600))
        display(pd.DataFrame([{
            "order": candidate.get("order", index),
            "bbox": candidate["bbox"],
            "decision": candidate["decision"],
            "method": candidate["method"],
        } for index, candidate in enumerate(page["candidates"], 1)]))
print("Panel catalog:", RUN_DIR / "panel_catalog.json")
print("Output directory:", RUN_DIR)

### Confirm or edit candidates

Edit `manifest["pages"][...]["candidates"]` below. A candidate is:
`{"bbox": [left, top, right, bottom], "decision": "accept", "order": 1, "method": "manual_review"}`.
The review must include every page; a page may have no candidates. Confirmation re-crops the images
and re-extracts descriptions before matching.

In [ ]:
if manifest:
    # Example manual box:
    # manifest["pages"][0]["candidates"].append({
    #     "bbox": [100, 200, 900, 700], "decision": "accept",
    #     "order": 99, "method": "manual_review"})
    reviewed_pages = [{"page_index": page["page_index"],
                       "candidates": [{**candidate, "order": candidate.get("order", index)}
                                      for index, candidate in enumerate(page["candidates"], 1)]}
                      for page in manifest["pages"]]
    prepared = apply_panel_review(RUN_DIR, reviewed_pages)
panel_catalog = pd.DataFrame(prepared["panels"])
catalog_fields = [key for key in ("evidence_index", "page_index", "panel_index",
                                   "panel_path", "review_status") if key in panel_catalog]
display(panel_catalog[catalog_fields])

## 3. Declare correspondence ranges

Use `evidence_index` from the panel catalog. A rule may cover several panels.
Times use the original video timeline, with half-open intervals `[start, end)`.
These are a declared comparison baseline, not verified Ground Truth and not retrieval constraints.

In [ ]:
# Edit these ranges or edit correspondence_ranges in config.local.json.
# Storyboard example:
# {"evidence_type": "storyboard", "panel_indices": [1, 2],
#  "final_start_sec": 45.0, "final_end_sec": 52.0}
# Animatic example:
# {"evidence_type": "animatic", "evidence_start_sec": 10.0, "evidence_end_sec": 18.0,
#  "final_start_sec": 45.0, "final_end_sec": 52.0}
CORRESPONDENCE_RANGES = config.get("correspondence_ranges", [])
config["correspondence_ranges"] = CORRESPONDENCE_RANGES
save_json(RUN_DIR / "config.json", config)
display(pd.DataFrame(CORRESPONDENCE_RANGES))

## 4. Global matching and automatic decisions

Visual CLIP ranks candidates across all reviewed evidence. Grouped text and valid local structure
provide Storyboard support; neighbouring candidates provide additional context.
Top-1 requires minimum visual similarity, a sufficient margin when a runner-up exists,
and auxiliary support. Excessive continuous reuse requires review.
Thresholds are configurable heuristics; validate their accuracy before trusting automation.
No monotonic path is imposed before sequence assessment.
Selected pairs are compared with the declared correspondence only after retrieval.

In [ ]:
result = retrieve(config, prepared)
decisions = pd.DataFrame(result["decisions"])
display(decisions)
display(decisions.groupby(["route", "status"]).size().rename("count").reset_index())

## 5. Inspect results and optionally correct uncertain cases

Only review cases you need to correct. A human choice overrides one Final/route.
The candidate ID must belong to that exact Final/route. Leave `HUMAN_REVIEWS` empty for fully automatic execution.

In [ ]:
candidate_table = pd.DataFrame(result["candidates"])
display(candidate_table)

def show_candidate(candidate_id):
    row = next(r for r in result["candidates"] if r["id"] == candidate_id)
    print("Final", row["final_index"], "Evidence", row["evidence_index"])
    display(DisplayImage(filename=row["final_path"], width=480))
    display(DisplayImage(filename=row["evidence_path"], width=480))
    print(row.get("description") or "No extracted description")

# show_candidate("final_storyboard:1:2")
HUMAN_REVIEWS = []
# Example:
# HUMAN_REVIEWS = [{"route": "final_storyboard", "final_index": 1,
#                  "decision": "matched", "candidate_id": "final_storyboard:1:2",
#                  "reason": "Pose and composition agree"}]
scores = apply_reviews(RUN_DIR, HUMAN_REVIEWS if HUMAN_REVIEWS else None)
display(pd.DataFrame(scores["provisional_and_human"]["route_order"]))
print(json.dumps(scores, indent=2))

## 6. Native provenance and timestamp comparisons

Inspect each submitted asset. Four provenance indicators retain their independent check-level rubrics.
Timestamp comparisons are separate. No aggregate provenance authenticity score is calculated.

In [ ]:
provenance_summary = inspect_provenance(config, prepared)
display(pd.DataFrame(provenance_summary).T[["status", "coverage"]])
provenance_scores = json.loads((RUN_DIR / "provenance_scores.json").read_text())
for asset, item in provenance_scores["assets"].items():
    print(asset)
    display(pd.DataFrame(item["indicator_scores"]).T[["label", "score", "reason"]])
display(json.loads((RUN_DIR / "timestamp_comparisons.json").read_text()))

## 7. Saved outputs

`matched_list.json` includes the decision source; automatic matches remain provisional.
`unmatched_list.json` means insufficient supporting evidence, while `pending_list.json` contains unresolved cases.
Human-confirmed metrics and provisional-plus-human metrics are reported separately.

In [ ]:
for name in ("matched_list", "unmatched_list", "pending_list"):
    path = RUN_DIR / (name + ".json")
    rows = json.loads(path.read_text())
    print(name, len(rows), path)
print("All JSON files:")
for path in sorted(RUN_DIR.glob("*.json")):
    print(path.name)